# Customer RFM Segmentation & Lifetime Value Analysis
**Author:** Aaryaraj Patil  
**Concepts:** Recency, Frequency, Monetary (RFM) Framework, Customer Lifetime Value (CLV), Marketing Strategy.

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = os.path.join('..', 'data', 'Brazilian E-Commerce Public Dataset by Olist')
orders = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
cust = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))
payments = pd.read_csv(os.path.join(data_dir, 'olist_order_payments_dataset.csv'))

print('Datasets loaded successfully.')

## 1. Feature Aggregation at the Unique Customer Level
In Olist, `customer_id` is generated per order, while `customer_unique_id` identifies the individual buyer.

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['order_purchase_timestamp'] = pd.to_datetime(delivered['order_purchase_timestamp'])

# Merge orders with unique customer ID and payment values
cust_orders = pd.merge(delivered, cust[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id')
order_pay = payments.groupby('order_id')['payment_value'].sum().reset_index()
full_df = pd.merge(cust_orders, order_pay, on='order_id', how='left')
full_df['payment_value'] = full_df['payment_value'].fillna(0.0)

snapshot_date = full_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = full_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'payment_value': 'sum'
}).reset_index()

rfm.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary']
rfm.describe()

## 2. RFM Scoring & Customer Segmentation

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['f_score'] = rfm['frequency'].apply(lambda f: 5 if f >= 4 else (4 if f == 3 else (3 if f == 2 else 1)))
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

def assign_rfm_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f == 1:
        return 'New Promising Customers'
    elif r == 3 and f == 1:
        return 'Potential Loyalists'
    elif r == 2 and f >= 3:
        return 'At Risk'
    elif r == 1 and f >= 3:
        return "Can't Lose Them"
    elif r <= 2 and f == 1 and m >= 4:
        return 'Hibernating High Spenders'
    elif r <= 2 and f == 1 and m < 4:
        return 'Lost / Inactive'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(assign_rfm_segment, axis=1)

segment_summary = rfm.groupby('segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_spend=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).reset_index()

segment_summary['revenue_pct'] = (segment_summary['total_revenue'] / segment_summary['total_revenue'].sum()) * 100
segment_summary.sort_values(by='total_revenue', ascending=False)

## 3. Executive Business Strategy by RFM Segment
- **Champions:** Reward with VIP perks, early product drops, and exclusive bundle discounts.
- **Hibernating High Spenders:** High lifetime spend but high recency. Trigger targeted win-back emails with personalized recommendations.
- **New Promising Customers:** High recent activity; introduce cross-selling campaigns within 30 days of first purchase to drive repeat orders.